# Day 6 - LoRA

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_6
- paper_ids: lora_2021
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

거대한 VLM 전체 weight를 업데이트하지 않고 어떤 low-rank parameter만 학습하며, rank와 target module은 비용과 표현력을 어떻게 바꾸는가?

## 2. Background theory

frozen linear weight `W`에 `Delta W = B A * alpha/r`를 더한다. `A[r,in]`, `B[out,r]`만 학습하므로 dense `out*in` 대신 `r*(in+out)` parameter가 필요하다. PEFT 기본 초기화는 B를 0으로 두어 시작 시 base model과 동일하게 만든다.

## 3. Paper connection

LoRA는 low intrinsic-rank update 가정으로 parameter-efficient fine-tuning을 한다. Qwen3-VL 공식 fine-tuning 코드는 attention의 `q_proj,k_proj,v_proj,o_proj`를 LoRA target으로 제시한다.

## 4. Input/output and shapes

입력 activation `[B,L,in]`; A를 지나 `[B,L,r]`; B를 지나 `[B,L,out]`; base linear output에 scaled delta를 더한다. adapter file만 저장해도 base checkpoint ID가 반드시 함께 필요하다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

4096x4096 dense layer와 rank 8 LoRA의 parameter 수를 비교하고 작은 행렬에서 delta를 계산한다.

In [ ]:
from vlm_foundation.lora import lora_delta, lora_fraction, lora_parameter_count

print("trainable params:", lora_parameter_count(4096, 4096, rank=8))
print("dense 대비 fraction:", lora_fraction(4096, 4096, rank=8))
x = np.array([[1.0, 2.0]])
a = np.array([[1.0, 0.0]])
b = np.array([[2.0], [3.0]])
print("delta:", lora_delta(x, a, b, alpha=1))

## 6. Visualization sanity check

B=0 initialization이면 delta가 정확히 0인지 확인한다. 그렇지 않으면 학습 전부터 base output이 달라진다.

In [ ]:
zero_b = np.zeros((2, 1))
assert np.allclose(lora_delta(x, a, zero_b, alpha=1), 0)
print("identity-at-initialization check: PASS")

## 7. Experiment

rank 4/8/16/64의 parameter fraction을 비교한다. rank만 늘리지 말고 validation metric과 overfitting을 함께 본다.

In [ ]:
for rank in [4, 8, 16, 64]:
    print(rank, lora_parameter_count(4096, 4096, rank), f"{100*lora_fraction(4096,4096,rank):.3f}%")

## 8. Metrics

trainable%, GPU peak memory, step time, train/validation loss, task metric, base capability regression을 기록한다.

## 9. Interpretation

LoRA는 memory를 줄이지만 activation과 frozen base weight는 여전히 필요하다. trainable parameter가 작다고 전체 학습 memory가 같은 비율로 줄지는 않는다.

## 10. Failure cases

target module 이름 mismatch, vision/projector가 완전히 frozen돼 domain gap을 못 줄이는 경우, 너무 높은 LR, adapter/base revision 불일치를 확인한다.

## 11. Real-service implications

여러 작은 adapter를 base model 하나에 교체 적용할 수 있다. adapter registry에 base model revision, prompt format, target modules, rank를 기록한다.

## 12. Review questions

1. LoRA parameter 수 공식은?
2. B=0 초기화의 의미는?
3. alpha/r scaling은 무엇을 조절하는가?
4. target module 선택이 중요한 이유는?
5. LoRA가 줄이지 못하는 memory는 무엇인가?